In [4]:
"""Automatic Chrome Dino launcher. Python standard library only."""
import argparse
import base64
import ctypes
import hashlib
import json
import os
from pathlib import Path
import shutil
import socket
import struct
import subprocess
import tempfile
import time
import urllib.request
from urllib.parse import urlsplit


class CDP:
    def __init__(self, url):
        u = urlsplit(url)
        if u.hostname not in ('127.0.0.1', 'localhost'):
            raise ValueError('只允許本機 Chrome 連線')
        self.sock = socket.create_connection((u.hostname, u.port), timeout=5)
        self.reader = self.sock.makefile('rb')
        key = base64.b64encode(os.urandom(16)).decode()
        self.sock.sendall(('GET '+u.path+' HTTP/1.1\r\nHost: '+u.netloc+
                          '\r\nUpgrade: websocket\r\nConnection: Upgrade\r\nSec-WebSocket-Key: '+key+
                          '\r\nSec-WebSocket-Version: 13\r\n\r\n').encode())
        status = self.reader.readline()
        headers = {}
        while True:
            line = self.reader.readline()
            if line in (b'\r\n', b''):
                break
            k, v = line.decode().split(':', 1)
            headers[k.lower()] = v.strip()
        expected = base64.b64encode(hashlib.sha1((key+'258EAFA5-E914-47DA-95CA-C5AB0DC85B11').encode()).digest()).decode()
        if b' 101 ' not in status or headers.get('sec-websocket-accept') != expected:
            self.close()
            raise RuntimeError('Chrome 偵錯連線失敗')
        self.serial = 0

    def close(self):
        self.reader.close()
        self.sock.close()

    def send(self, payload, opcode=1):
        mask = os.urandom(4)
        n = len(payload)
        header = bytes([0x80 | opcode])
        header += bytes([0x80 | n]) if n < 126 else (bytes([0x80 | 126])+struct.pack('!H', n) if n < 65536 else bytes([0x80 | 127])+struct.pack('!Q', n))
        self.sock.sendall(header+mask+bytes(v ^ mask[i % 4] for i, v in enumerate(payload)))

    def read(self, n):
        data = self.reader.read(n)
        if len(data) != n:
            raise ConnectionError('Chrome 視窗已關閉')
        return data

    def receive(self):
        message = bytearray()
        while True:
            a, b = self.read(2)
            length = b & 127
            if length == 126:
                length = struct.unpack('!H', self.read(2))[0]
            elif length == 127:
                length = struct.unpack('!Q', self.read(8))[0]
            if length > 16*1024*1024:
                raise RuntimeError('回應過大')
            mask = self.read(4) if b & 128 else None
            payload = self.read(length)
            if mask:
                payload = bytes(v ^ mask[i % 4] for i, v in enumerate(payload))
            opcode = a & 15
            if opcode == 8:
                raise ConnectionError('Chrome 視窗已關閉')
            if opcode == 9:
                self.send(payload, 10)
                continue
            if opcode == 10:
                continue
            message.extend(payload)
            if a & 128:
                return json.loads(message)

    def call(self, method, **params):
        self.serial += 1
        self.send(json.dumps(dict(id=self.serial, method=method, params=params)).encode())
        while True:
            result = self.receive()
            if result.get('id') == self.serial:
                if 'error' in result:
                    raise RuntimeError(str(result['error']))
                return result.get('result', {})

    def evaluate(self, expression):
        result = self.call('Runtime.evaluate', expression=expression, awaitPromise=True, returnByValue=True)
        if 'exceptionDetails' in result:
            raise RuntimeError(str(result['exceptionDetails']))
        return result.get('result', {}).get('value')


def find_chrome(explicit=None):
    candidates = [explicit, shutil.which('chrome')]
    for base in ('LOCALAPPDATA', 'PROGRAMFILES', 'PROGRAMFILES(X86)'):
        candidates.append(str(Path(os.environ.get(base, ''))/'Google/Chrome/Application/chrome.exe'))
    for path in candidates:
        if path and Path(path).is_file():
            return path
    raise FileNotFoundError('找不到 Chrome。可用 --chrome 指定 chrome.exe 路徑。')


INIT = r'''(async () => {
 let R = typeof Runner !== 'undefined' ? Runner : globalThis.Runner;
 if (!R) R = (await import('/dino_game/offline.js')).Runner;
 const r = R.getInstance ? R.getInstance() : R.instance_;
 if (!r || !r.tRex || !r.horizon || typeof r.tRex.startJump !== 'function')
   throw new Error('此 Chrome 的遊戲介面不相容，或學校已停用小恐龍');
 globalThis.__pythonDino = r;
 return true;
})()'''
STATE = '''(() => { const r=globalThis.__pythonDino, t=r.tRex;
 return {crashed:r.crashed, playing:r.playing, jumping:t.jumping, ducking:t.ducking,
 speed:r.currentSpeed, x:t.xPos, width:t.config.width || t.config.WIDTH || 44,
 y:t.yPos, ground:t.groundYPos, velocity:t.jumpVelocity,
 distance:r.distanceRan, obstacles:r.horizon.obstacles.map(o=>({x:o.xPos,width:o.width,y:o.yPos,type:o.typeConfig.type,
 levels:Array.isArray(o.typeConfig.yPos)?o.typeConfig.yPos:null}))}; })()'''


def obstacle_action(obstacle):
    if obstacle['type'].lower() != 'pterodactyl':
        return 'jump'
    levels = sorted(set(obstacle.get('levels') or [50, 75, 100]))
    index = min(range(len(levels)), key=lambda i: abs(levels[i]-obstacle['y']))
    if index == 0:
        return 'ignore'
    if len(levels) >= 3 and index == 1:
        return 'duck'
    return 'jump'


def decide_action(state, lead):
    if not state['playing'] or state['jumping']:
        return 'wait'
    obstacles = sorted((o for o in state['obstacles']
                        if o['x']+o['width'] > state['x']-8
                        and obstacle_action(o) != 'ignore'), key=lambda o: o['x'])
    if not obstacles:
        return 'run'
    obstacle = obstacles[0]
    action = obstacle_action(obstacle)
    distance = obstacle['x']-state['x']-state['width']
    seconds = max(lead, .2) if action == 'duck' else lead
    return action if distance <= state['speed']*60*seconds else 'run'


def main():
    parser = argparse.ArgumentParser(description='自動開啟 Chrome 並操作小恐龍，不需要安裝套件')
    parser.add_argument('--chrome', help='Chrome 執行檔路徑')
    parser.add_argument('--lead', type=float, default=.12, help='跳躍提前秒數（預設 0.12）')
    parser.add_argument('--seconds', type=float, default=0, help='測試時間，0 表示持續執行')
    parser.add_argument('--debug', action='store_true', help='記錄跳躍與碰撞狀態')
    
    # 改用 parse_known_args() 忽略 Jupyter 傳入的額外參數
    args, _ = parser.parse_known_args()

    if not .05 <= args.lead <= .5 or args.seconds < 0:
        parser.error('lead 需介於 0.05–0.5，seconds 不可為負數')
    chrome = find_chrome(args.chrome)
    profile = Path(tempfile.mkdtemp(prefix='python-dino-'))
    connection = None
    process = None
    try:
        print('開啟獨立 Chrome 小恐龍視窗；F8 或 Ctrl+C 停止。', flush=True)
        process = subprocess.Popen([chrome, '--user-data-dir='+str(profile),
            '--remote-debugging-port=0', '--remote-debugging-address=127.0.0.1',
            '--no-first-run', '--no-default-browser-check', '--new-window',
            '--start-fullscreen', 'chrome://dino/'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        last_error = None
        deadline = time.monotonic()+25
        opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))
        while time.monotonic() < deadline:
            if ctypes.windll.user32.GetAsyncKeyState(0x77) & 0x8000:
                raise KeyboardInterrupt
            try:
                port = int((profile/'DevToolsActivePort').read_text().splitlines()[0])
                with opener.open('http://127.0.0.1:%d/json/list' % port, timeout=1) as response:
                    pages = json.load(response)
                page = next(p for p in pages if p.get('type') == 'page')
                connection = CDP(page['webSocketDebuggerUrl'])
                if not page.get('url', '').startswith('chrome://dino'):
                    connection.call('Page.navigate', url='chrome://dino/')
                    time.sleep(.5)
                connection.evaluate(INIT)
                break
            except (OSError, ValueError, StopIteration, RuntimeError) as error:
                last_error = error
                if connection:
                    connection.close()
                    connection = None
                time.sleep(.3)
        if not connection:
            raise RuntimeError('25 秒內無法連接小恐龍；可能是 Chrome 版本或學校政策不允許遠端偵錯。原因：'+repr(last_error))
        connection.call('Page.bringToFront')
        connection.call('Input.dispatchKeyEvent', type='keyDown', key=' ', code='Space', windowsVirtualKeyCode=32)
        connection.call('Input.dispatchKeyEvent', type='keyUp', key=' ', code='Space', windowsVirtualKeyCode=32)
        print('已自動開始。不需選座標；由 Python 判斷跳躍。', flush=True)
        started = last_report = time.monotonic()
        dead_since = None
        while not args.seconds or time.monotonic()-started < args.seconds:
            if ctypes.windll.user32.GetAsyncKeyState(0x77) & 0x8000:
                break
            state = connection.evaluate(STATE)
            now = time.monotonic()
            if state['crashed']:
                if dead_since is None:
                    print('碰撞：'+(json.dumps(state) if args.debug else
                          '距離 {:.0f}，稍後自動重開'.format(state['distance'])), flush=True)
                dead_since = dead_since or now
                if now-dead_since > 1.5:
                    connection.evaluate('globalThis.__pythonDino.restart()')
                    dead_since = None
            else:
                dead_since = None
                action = decide_action(state, args.lead)
                if action == 'duck' and not state['ducking']:
                    connection.evaluate('globalThis.__pythonDino.tRex.setDuck(true)')
                    if args.debug:
                        print('蹲下：'+json.dumps(state), flush=True)
                elif action in ('jump', 'run'):
                    if state['ducking']:
                        connection.evaluate('globalThis.__pythonDino.tRex.setDuck(false)')
                    if action == 'jump':
                        if args.debug:
                            print('跳躍：'+json.dumps(state), flush=True)
                        connection.evaluate('globalThis.__pythonDino.tRex.startJump(globalThis.__pythonDino.currentSpeed)')
            if now-last_report >= 5:
                print('速度 {:.1f}；距離 {:.0f}；{}'.format(state['speed'], state['distance'], '碰撞後重開' if state['crashed'] else '運行中'), flush=True)
                last_report = now
            time.sleep(.015)
    finally:
        if connection:
            try:
                connection.call('Browser.close')
            except (OSError, RuntimeError):
                pass
            connection.close()
        if process:
            try:
                process.wait(timeout=5)
            except subprocess.TimeoutExpired:
                process.terminate()
                process.wait(timeout=5)
        # Only remove the uniquely created temporary profile, never user profiles.
        shutil.rmtree(profile, ignore_errors=True)


if __name__ == '__main__':
    try:
        main()
    except KeyboardInterrupt:
        print('已停止。')
    except (OSError, RuntimeError, ValueError) as error:
        print('無法執行：', error)
        raise SystemExit(1)

開啟獨立 Chrome 小恐龍視窗；F8 或 Ctrl+C 停止。
已自動開始。不需選座標；由 Python 判斷跳躍。
速度 7.4；距離 1811；運行中
速度 8.9；距離 4258；運行中
速度 10.3；距離 7137；運行中
速度 11.8；距離 10449；運行中
速度 13.0；距離 14194；運行中
速度 13.0；距離 18094；運行中
速度 13.0；距離 21989；運行中
速度 13.0；距離 25889；運行中
速度 13.0；距離 29789；運行中
速度 13.0；距離 33689；運行中
速度 13.0；距離 37595；運行中
速度 13.0；距離 41495；運行中
速度 13.0；距離 45395；運行中
速度 13.0；距離 49290；運行中
速度 13.0；距離 53196；運行中
速度 13.0；距離 57101；運行中
速度 13.0；距離 60991；運行中
速度 13.0；距離 64896；運行中
速度 13.0；距離 68791；運行中
速度 13.0；距離 72697；運行中
速度 13.0；距離 76591；運行中
速度 13.0；距離 80497；運行中
速度 13.0；距離 84392；運行中
速度 13.0；距離 88303；運行中
速度 13.0；距離 92192；運行中
速度 13.0；距離 96098；運行中
速度 13.0；距離 100003；運行中
速度 13.0；距離 103898；運行中
速度 13.0；距離 107798；運行中
速度 13.0；距離 111693；運行中
速度 13.0；距離 115594；運行中
速度 13.0；距離 119505；運行中
速度 13.0；距離 123400；運行中
速度 13.0；距離 127294；運行中
速度 13.0；距離 131200；運行中
速度 13.0；距離 135105；運行中
速度 13.0；距離 139000；運行中
速度 13.0；距離 142901；運行中
速度 13.0；距離 146807；運行中
速度 13.0；距離 150707；運行中
速度 13.0；距離 154596；運行中
速度 13.0；距離 158502；運行中
速度 13.0；距離 162407；運行中
速度 13.0；距離 166308；運行中
速度

# Python 自動遊玩 Chrome Dino（小恐龍）程式開發對話紀錄

---

### **使用者**
幫我寫一個自動遊玩小恐龍的程式

---

### **AI**
沒問題！要寫自動遊玩 Chrome Dino（小恐龍）程式，通常有兩種常見做法：
1. **影像辨識 + 模擬鍵盤**：用 `pyautogui` 擷取螢幕畫面，偵測到前方有障礙物時自動按空白鍵跳躍。
2. **Chrome 偵錯協定 (CDP) + JS 注入**：透過 WebSocket 直接讀取遊戲內部的真實狀態（速度、障礙物座標），並用 JS 觸發跳躍。

請問你想用哪一種方式？

---

### **使用者**
先用第一種方法試試看！我想先寫一個簡單好理解的，用影像辨識怎麼做？

---

### **AI**
好的！「影像辨識 + 模擬鍵盤」的原理非常直觀，主要分為三個步驟：

1. **定位檢測區域（ROI）**：在小恐龍前方設定一個小方塊區域（例如恐龍前方 50~100 像素的位置）。
2. **像素顏色變化偵測**：預設背景是白色（或夜間模式的黑色）。持續抓取該區域的像素，只要發現顏色變深（代表仙人掌或鳥飛過來了），就觸發動作。
3. **模擬按鍵**：呼叫 `pyautogui.press('space')` 讓恐龍跳起來。

不過這種方法有一個缺點：隨著遊戲速度越來越快，固定的檢測區域會來不及反應，而且夜間模式切換時容易誤判。

---

### **使用者**
原來如此！那如果想要克服速度變快跟夜間模式，第二種 CDP 的方法是怎麼做的？可以改用純 Python 標準函式庫寫嗎？

---

### **AI**
沒問題！第二種方法是用 **Chrome DevTools Protocol (CDP)**，它完全不需要安裝任何第三方套件（純 Python 標準函式庫即可實現）：

1. **啟動遠端偵錯**：用 `subprocess` 啟動獨立 Chrome 視窗，並帶上 `--remote-debugging-port=0` 參數開啟偵錯通道。
2. **建立 WebSocket 連線**：用 Python 內建的 `socket` 手動完成 WebSocket 握手，直接與 Chrome 的 CDP 通訊。
3. **注入 JavaScript 讀取狀態**：直接向遊戲注入 JS，毫秒級讀取恐龍位置、目前速度、前方障礙物的精確 X/Y 座標與種類。

這樣就不受螢幕解析度、視窗位置或夜間模式影響，反應速度極快！

---

### **使用者**
障礙物好像有分仙人掌跟不同高度的翼龍，Python 該怎麼判斷要跳過還是蹲下？

---

### **AI**
透過 CDP 抓到障礙物數據後，我們可以寫一個決策邏輯：

* **仙人掌 / 低空翼龍**：直接跳躍 (`jump`)。
* **中空翼龍**：這高度站著會撞到、跳起來也會撞到，必須發送 `setDuck(true)` 蹲下 (`duck`)。
* **高空翼龍**：高度高於恐龍，直接無視 (`ignore`)。

接著根據目前速度計算門檻：當 `障礙物距離 <= 當前速度 * 60 * 提前秒數 (lead)` 時，立刻發送對應的跳躍或蹲下指令。

---

### **使用者**
如果在 Windows 上執行，要怎麼支援自動搜尋 Chrome 路徑，還有緊急停止功能？另外我想讓它能在 Jupyter Notebook 執行。

---

### **AI**
這幾個需求可以這樣處理：

1. **自動搜尋 Chrome**：檢查 `LOCALAPPDATA` 與 `PROGRAMFILES` 等環境變數下的 `Google/Chrome/Application/chrome.exe` 路徑。
2. **緊急停止**：使用 `ctypes.windll.user32.GetAsyncKeyState(0x77)` 監聽按鍵，隨時按下 **F8** 或 **Ctrl+C** 即可安全結束程式並關閉視窗。
3. **Jupyter 相容**：解析命令列參數時改用 `parser.parse_known_args()`，避免 Jupyter 傳入的額外參數導致程式報錯。

整合這些邏輯後，就能寫出一支完全免安裝套件、開箱即用的自動小恐龍腳本了！